# Project name - FMCG Sales Drivers Analysis

## Statistical and Quasi-Causal Analysis of Sales Drivers in FMCG Retail: A Multivariate Study

# 1. Problem Definition (Scientific Method)

## Research Question
    This project aims to answer the following core question:
    What affects sales?
    Which operational, environmental, and store-level factors significantly drive FMCG beverage sales?

    More specifically, we investigate which operational, environmental, and store-level factors significantly influence sales performance.
    
## Business & Analytical Motivation
    Understanding sales drivers is critical for FMCG companies because it enables:

    - Improved demand forecasting
    - Optimized field operations (store visits)
    - Better allocation of refrigeration and branding equipment
    - Data-driven retail strategy decisions
    - Increased revenue efficiency across stores and regions

    This project bridges business intelligence and statistical modeling to identify both correlation and causal relationships.

## Hypotheses
    We test the following hypotheses using statistical and machine learning methods:
    H1: Store Visits Effect
        - Store visits have a positive and statistically significant impact on sales.
    H2: Weather Impact
         - Weather conditions (temperature, precipitation, humidity) significantly influence beverage demand.
    H3: Store Execution Effect
        - Stores with better execution (equipment, branding, availability) generate higher sales.
    H4: Heterogeneity Across Stores
        - The effect of operational and environmental drivers varies across: 
        regions
        store segments
        trade channels

## Causality Considerations

    While the project aims to identify drivers of sales, it is important to distinguish between correlation and causation.

    Baseline statistical models (correlation, OLS) identify associations, but may suffer from:
    - Endogeneity
    - Omitted variable bias
    - Selection bias (e.g., high-performing stores may receive more visits)

    To address this, we extend the analysis using:
    - Lagged variables (temporal ordering)
    - Fixed Effects models (controlling for unobserved heterogeneity)

    These approaches improve causal interpretability, though results should still be interpreted with caution.

# 2. DATA ARCHITECTURE
    The project integrates multiple data sources into a unified analytical dataset.
    
## CORE DATA (Business Layer)
### 1. SALES (target dataset)
    Contains daily transactional performance per store:
    date
    customer (store ID)
    revenue_bgn (sales value in BGN)
    cases (volume sold)

    This is the main dependent variable used in all models.
## OPERATIONS LAYER
### 2. VISITS 
    Represents field activity:
    Customer
    date
    visits (number of executed store visits)

    Measures sales force intensity and retail engagement.
### 3. COOLER / EQUIPMENT
    Represents in-store execution quality:
    customer
    equipment count
    branding
    status

    Captures availability and visibility of products in-store.
## MASTER STORE DATA
### 4. LIST DATASET
    Provides static store characteristics:
    customer
    region
    city
    channel
    segment

    Explains structural differences between stores.
## EXTERNAL ENVIRONMENT
### 5. WEATHER
    Daily environmental conditions per region:
    date
    temperature (°C)
    precipitation (mm)
    humidity (%)
    wind (km/h)
    holiday
    weekend
    non working day

    Captures external demand-driving conditions.

# 3. DATA INGESTION & INITIAL INSPECTION

Data Loading Strategy (Multi-source ingestion)

The dataset is constructed using multiple heterogeneous data sources including sales, store information, operational visits, equipment data, and external weather data.

Each dataset is loaded independently from cloud storage (Google Drive) to ensure modularity and reproducibility of the pipeline.

This approach separates raw data ingestion from transformation logic,
improving maintainability and debuggability.

### Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import holidays
import time
import statsmodels.api as sm
from scipy.stats import f_oneway
from scipy.stats import ttest_ind

### DATA LOADING - RAW LAYER

In [2]:
# LOAD FROM GOOGLE DRIVE IDS

def load_drive(file_id, compression=None):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    return pd.read_csv(url, compression=compression)
    
def load_data():
    sales = load_drive("1g-2KlxWBgJwlVULMSh1jmXvPrzgpI0tJ", compression="gzip")
    visits = load_drive("1JlVGwR1koth70guNf8fygJx207qhn9eZ")
    store = load_drive("1gC53aiUNdGl5Za2Y0kr-iDW6-UPQWs1d")
    cooler = load_drive("18M8gADRTypnnqWZkjQnkhVcl3Y6H1AGg")
    return sales, visits, store, cooler

### INITIAL DATA INSPECTION

In [3]:
datasets = {
    "sales": sales,
    "visits": visits,
    "cooler": cooler,
    "store": store
}

for name, df in datasets.items():
    print("\n", "="*40)
    print(name.upper())
    print(df.head())
    print(df.info())
    print("Shape:", df.shape)

NameError: name 'sales' is not defined

# 4. DATA CLEANING & STANDARDIZATION

After loading, all datasets undergo a standardized preprocessing pipeline to ensure structural consistency across sources.

This includes:

    - Standardization of column names (removal of hidden characters and formatting inconsistencies)
    - Harmonization of naming conventions across datasets (e.g., Customer → customer)
    - Conversion of numerical fields stored as strings into numeric format (e.g., revenue, cases)
    - Transformation of date fields into datetime format for downstream processing

This step ensures all datasets share a consistent schema prior to integration.

Time Alignment

All datasets are aligned at daily granularity using a unified date key (date),
ensuring temporal consistency and enabling reliable cross-source joins.

### COLUMN CLEANING

In [ ]:
# remove hidden characters BEFORE everything else
def clean_cols(df):
    df.columns = (
        df.columns
        .str.replace("\n", " ")
        .str.replace("\r", " ")
        .str.strip()
    )
    return df

sales = clean_cols(sales)
visits = clean_cols(visits)
cooler = clean_cols(cooler)
store = clean_cols(store)

#print(df.head())

### RENAME COLUMNS - STANDARDIZATION LAYER

In [ ]:
visits = visits.rename(columns={
    "Calendar day": "date",
    "Customer": "customer",
    "Executed Visits All": "visits"
})

store = store.rename(columns={
    "Customer": "customer"
})

cooler = cooler.rename(columns={
    "Customer": "customer",
    "Number of Equipments": "equipment_count"
})

### TYPE CONVERSION 

In [ ]:
def clean_sales(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["revenue_bgn"] = (
        df["revenue_bgn"].astype(str).str.replace(",", ".").astype(float)
    )
    df["cases"] = pd.to_numeric(df["cases"], errors="coerce")
    return df

In [ ]:
# SALES
sales = clean_sales(sales)

# VISITS
visits["date"] = pd.to_datetime(visits["date"], errors="coerce", dayfirst=True)
visits["visits"] = pd.to_numeric(visits["visits"], errors="coerce")

### DATA QUALITY CHECK 

In [ ]:
print("SALES:", sales.columns.tolist())
print("VISITS:", visits.columns.tolist())
print("STORE:", store.columns.tolist())
print("COOLER:", cooler.columns.tolist())

### WEATHER PIPELINE (EXTERNAL DATA SOURCE)

In [ ]:
def get_weather(city, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "windspeed_10m_max"
        ],
        "timezone": "Europe/Sofia"
    }

    r = requests.get(url, params=params)
    data = r.json()["daily"]

    df = pd.DataFrame(data)
    df["city"] = city
    return df


cities = {
     "Varna": (43.2141, 27.9147),
    "Burgas": (42.5048, 27.4626),
    "Shumen": (43.2712, 26.9361),
    "Ruse": (43.8356, 25.9657),
    "Sliven": (42.6818, 26.3227),
    "Veliko Tarnovo": (43.0757, 25.6172),
    "Yambol": (42.4840, 26.5030),
    "Targovishte": (43.2512, 26.5729),
    "Razgrad": (43.5333, 26.5167),
    "Dobrich": (43.5667, 27.8333),
    "Silistra": (44.1167, 27.2667),
    "Stara Zagora": (42.4258, 25.6345),
    "Plovdiv": (42.1354, 24.7453),
    "Pleven": (43.4170, 24.6067),
    "Gabrovo": (42.8747, 25.3342)
}

all_weather = []

for city, (lat, lon) in cities.items():
    print("Downloading:", city)
    df = get_weather(city, lat, lon)
    all_weather.append(df)
    time.sleep(1)
    
print("Download completed.")

weather = pd.concat(all_weather, ignore_index=True)

weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_celsius",
    "precipitation_sum": "precipitation_mm",
    "relative_humidity_2m_mean": "humidity_percent",
    "windspeed_10m_max": "wind_speed_kmh"
})

weather["date"] = pd.to_datetime(weather["date"])

# 5. DATA INTEGRATION & CROSS-SOURCE MAPPING

### Region–Weather Mapping Logic

Weather data is not directly available per store, but only per city.

However, store-level data contains regional identifiers in Bulgarian language, while weather dataset uses English city names.

To resolve this mismatch, a mapping layer is introduced:

    Bulgarian Region → English City Name mapping

Example:

    “Варна” → “Varna”
    “Бургас” → “Burgas”

This mapping ensures correct alignment between:

    Store location (business layer)
    Weather conditions (external environment layer)

In [ ]:
# REGION → CITY MAPPING 

region_to_city = {
    "Варна": "Varna",
    "Бургас": "Burgas",
    "Шумен": "Shumen",
    "Русе": "Ruse",
    "Сливен": "Sliven",
    "Велико Търново": "Veliko Tarnovo",
    "Ямбол": "Yambol",
    "Търговище": "Targovishte",
    "Разград": "Razgrad",
    "Добрич": "Dobrich",
    "Силистра": "Silistra",
    "Стара Загора": "Stara Zagora",
    "Пловдив": "Plovdiv",
    "Плевен": "Pleven",
    "Габрово": "Gabrovo"
}

In [ ]:
store["Region_en"] = store["Region"].map(region_to_city)

print(store.head())

Cooler AGGREGATION

In [ ]:
cooler_agg = cooler.groupby("customer").agg({
    "equipment_count": "sum",
    "Number of doors": "count", 
}).reset_index()

cooler_agg = cooler_agg.rename(columns={
    "Number of Equipments": "equipment_count",
    "Number of doors": "doors_count"
})

### DATA INTEGRATION & MASTER DATASET CONSTRUCTION

After standardization and mapping, all datasets are merged into a single analytical table using the following keys:

    Primary key: customer
    Temporal key: date
    Environmental join key: Region → City mapping

The final integration follows this order:

   1. Sales + Visits (customer + date)
   2. Store attributes (customer)
3. Equipment data (customer)
4. Weather data (date + mapped city)

This results in a unified panel dataset containing:

    Sales performance
    Operational intensity
    Store characteristics
    Environmental conditions

In [ ]:
df = sales.merge(visits, on=["customer", "date"], how="left")

df = df.merge(store, on="customer", how="left")

df = df.merge(cooler_agg, on="customer", how="left")

# WEATHER MERGE (correct alignment: date + city)
df = df.merge(
    weather,
    left_on=["date", "Region_en"],
    right_on=["date", "city"],
    how="left"
)

df = df.drop(columns=["Region_en", "city"])

### FEATURE ENGINEERING

We construct several derived variables:

    - Sales Efficiency Metrics
$$
\text{sales\_per\_case} = \frac{\text{revenue}}{\text{cases}} + 1
$$

$$
\text{sales\_per\_visit} = \frac{\text{revenue}}{\text{visits}} + 1
$$

    - Visit Intensity
$$
\text{high\_visit} = 1{\{\text{visits} > Q_{0.75}\}}
$$

    - Temperature Buckets
    Categorical discretization into cold/cool/warm/hot.
$$
T \in \{\text{cold}, \text{cool}, \text{warm}, \text{hot}\}
$$


In [ ]:
df["sales_per_case"] = df["revenue_bgn"] / (df["cases"].fillna(0) + 1)
df["sales_per_visit"] = df["revenue_bgn"] / (df["visits"].fillna(0) + 1)

df["high_visit"] = df["visits"] > df["visits"].quantile(0.75)

df["temperature_bucket"] = pd.cut(
    df["temperature_celsius"],
    bins=[-10, 10, 20, 30, 40],
    labels=["cold", "cool", "warm", "hot"]
)

df["is_active_store"] = (
    (df["visits"].fillna(0) > 0) |
    (df["revenue_bgn"].fillna(0) > 0)
).astype(int)

Advanced Feature Engineering

To capture more complex relationships, additional features are constructed:

1. Temporal Features
- Day of week (captures weekly patterns)
- Month (captures seasonality)

2. Lagged Variables
- Previous day visits
- Previous day sales

3. Interaction Effects
- Visits × Temperature

4. Non-linear Effects
- Squared temperature (captures non-linear demand)

These features allow modeling of dynamic, seasonal, and interaction-driven effects.

In [ ]:
# TIME FEATURES
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

# LAG FEATURES
df["visits_lag1"] = df.groupby("customer")["visits"].shift(1)
df["sales_lag1"] = df.groupby("customer")["revenue_bgn"].shift(1)

# INTERACTION
df["temp_x_visits"] = df["temperature_celsius"] * df["visits"]

# NON-LINEAR
df["temp_sq"] = df["temperature_celsius"] ** 2

### FINAL CHECK

Outcome of Pipeline

The final dataset enables:

    Time-series consistency across all variables
    Cross-sectional comparability across stores
    Correct alignment of weather effects by region
    Reliable basis for regression and causal analysis

In [ ]:
print(df.shape)
print(df.head())
print(df.isna().sum().head(10))

# --- if we want to save the date in .csv ---
# output_path = "weather_dataset_bg.csv"
# df.to_csv(output_path, index=False)

# 6. EXPLORATORY DATA ANALYSIS (EDA)

    The goal of the exploratory analysis is to identify patterns, distributions, and relationships between sales and potential explanatory variables.

    We perform:

    - Univariate Analysis

We analyze the distribution of key variables such as:

revenue (sales)
store visits
equipment availability

This helps identify skewness, outliers, and general data behavior.

    - Bivariate Analysis

We examine relationships between:

Store visits and sales
Temperature and sales
Equipment availability and sales

These relationships provide early evidence of potential causal effects.

    - Multivariate Analysis

We extend the analysis to multiple variables simultaneously by:

correlation matrix analysis
segmentation by store type and region
grouping stores based on performance levels

This allows identification of interaction effects and hidden patterns in the data.

Additional Exploratory Analysis

To better understand structure in the data, we extend EDA with:

- Sales distribution in log scale
- Sales by store segment (boxplots)
- Temporal trends (sales over time)
- Sales by temperature bucket

In [ ]:


# SALES DISTRIBUTION
df["revenue_bgn"].hist()
plt.title("Sales Distribution")
plt.show()

# VISITS vs SALES
plt.scatter(df["visits"], df["revenue_bgn"])
plt.title("Visits vs Sales")
plt.show()

# TEMPERATURE vs SALES
plt.scatter(df["temperature_celsius"], df["revenue_bgn"])
plt.title("Temperature vs Sales")
plt.show()

# 7. STATISTICAL INFERENCE

To move beyond descriptive analysis, we apply statistical inference methods to evaluate relationships between key variables and sales performance.

We use three complementary approaches:
- Correlation analysis (linear associations)
- Hypothesis testing (mean differences across groups)
- Variance analysis (heterogeneity across segments)

## Correlation Analysis

We first examine pairwise linear relationships using Pearson correlation coefficients.

The results indicate weak but positive relationships between sales and both visits and temperature, while no strong linear dependencies are observed overall.

In [ ]:
df[["revenue_bgn","visits","temperature_celsius"]].corr()

## T-Test Analysis

We test whether average sales differ significantly between high-visit and low-visit stores using an independent samples t-test.

This helps evaluate whether store visit intensity is associated with differences in sales performance.

We compare high-visit vs low-visit stores:

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$ 

Result: p < 0.0001, significant difference.

High visit stores vs low visit stores

Objective:

Determine whether store visits significantly affect sales performance.

In [ ]:
high = df[df["high_visit"] == True]["revenue_bgn"]
low = df[df["high_visit"] == False]["revenue_bgn"]

ttest_ind(high, low, nan_policy='omit')

## ANOVA Analysis

To assess heterogeneity across store segments, we perform a one-way ANOVA test.

This allows us to test whether mean sales differ significantly across categorical groups (Bronze, Silver, Gold).

We extend the comparison across multiple groups.

Testing differences across store segments:

$$
F = \frac{\text{Between-group variance}}{\text{Within-group variance}}
$$

Result: p = 0.0, strong evidence of heterogeneity.

Store segments (Bronze, Silver, Gold)
Regional performance groups

Objective:

Test whether differences in sales are statistically significant across categories.

In [ ]:
groups = [g["revenue_bgn"].dropna() for _, g in df.groupby("Segment")]
f_oneway(*groups)

## Hypothesis Evaluation

Each hypothesis is evaluated based on p-values and statistical significance thresholds (α = 0.05).

# 8. REGRESSION MODELING

To quantify the impact of operational and environmental factors on sales, we estimate a sequence of regression models with increasing complexity.

The model is defined as:

Sales = f(visits, temperature, humidity, equipment, store characteristics)

Model Objective:
Estimate the magnitude of each driver
Identify statistically significant predictors
Measure overall explanatory power (R²)
Interpretation:
Positive coefficients indicate drivers that increase sales
Negative coefficients indicate inhibitory effects
Statistical significance determines reliability of predictors

    1. Model Specification
We estimate:

$$
\log(y_{it}) = \beta_0 + \beta_1 \cdot visits_{it} + \beta_2 \cdot temperature_{rt} + \beta_3 \cdot humidity_{rt} + \beta_4 \cdot equipment_i + u_{it}
$$

Why log-transform?

    - Stabilizes variance
    - Reduces skewness
    - Coefficients become elasticities

    2. Results Summary

    - Temperature: positive, significant
    - Humidity: negative, significant
    - Equipment count: strong positive effect
    -Visits: not significant in OLS (but see causal model below)

Handling Missing Visits

A large portion of visit data is missing. Instead of removing these observations, missing values are interpreted as absence of recorded visits rather than true missingness.

Thus, missing visit values are imputed as zero to preserve dataset size and avoid selection bias.

## Baseline Model

We first estimate a simple OLS regression to measure the direct relationship between sales and key drivers.

This model provides an initial benchmark but does not account for time dynamics or non-linear effects.

In [ ]:
# CLEAN DATA
# махаме отрицателни и нулеви продажби (задължително за log)
df = df[df["revenue_bgn"] > 0].copy()

# махаме липсващи стойности за ключови променливи
df = df.dropna(subset=[
    "temperature_celsius",
    "humidity_percent",
    "equipment_count"
])

df["visits"] = df["visits"].fillna(0)

# TARGET VARIABLE
df["log_sales"] = np.log(df["revenue_bgn"])

# 3. FEATURES (X)
X = df[[
    "visits",
    "temperature_celsius",
    "humidity_percent",
    "equipment_count"
]]
# ако има NaN 
X = X.fillna(0)

X = sm.add_constant(X)
y = df["log_sales"]
model = sm.OLS(y, X, missing="drop").fit()

print(model.summary())

## Extended Model with Additional Features

We extend the baseline model by including lagged variables, interaction terms, and temporal controls to capture dynamic and non-linear effects.

This improves model flexibility and captures more realistic sales behavior over time.

In [ ]:
X = df[[
    "visits",
    "visits_lag1",
    "temperature_celsius",
    "temp_sq",
    "humidity_percent",
    "equipment_count",
    "temp_x_visits",
    "day_of_week",
    "month"
]]

X = X.fillna(0)
X = sm.add_constant(X)

model_ext = sm.OLS(y, X).fit()
print(model_ext.summary())

## Fixed Effects Model (Controlling for Unobserved Heterogeneity)

To address unobserved heterogeneity across stores, we estimate a fixed effects model using within-store variation.

This approach removes time-invariant differences such as store location and customer base, allowing us to isolate the effect of time-varying drivers on sales.

In [ ]:
# copy dataset safely
df_fe = df.copy()

# ensure no missing key columns
cols = [
    "log_sales",
    "visits",
    "visits_lag1",
    "temperature_celsius",
    "humidity_percent",
    "equipment_count"
]

df_fe = df_fe.dropna(subset=cols)

# -------------------------
# WITHIN TRANSFORMATION
# -------------------------

grouped = df_fe.groupby("customer")

df_fe["log_sales_dm"] = df_fe["log_sales"] - grouped["log_sales"].transform("mean")

X_vars = [
    "visits",
    "visits_lag1",
    "temperature_celsius",
    "humidity_percent",
    "equipment_count"
]

for col in X_vars:
    df_fe[col + "_dm"] = df_fe[col] - grouped[col].transform("mean")

# -------------------------
# MODEL
# -------------------------

import statsmodels.api as sm

X = df_fe[[col + "_dm" for col in X_vars]]
X = sm.add_constant(X)

y = df_fe["log_sales_dm"]

model = sm.OLS(y, X).fit(cov_type="HC3")

print(model.summary())

# 9. ADVANCED INSIGHTS

To deepen the analysis, we explore:

    1. Interaction Effects

We analyze whether combined effects exist between variables:

Visits * Weather conditions
Store execution * Store type

    2. Lag Effects

We test whether past operational activity influences future sales:

Previous visits -> current sales

    3. Seasonality Effects

We evaluate temporal patterns:

Weekend vs weekday behavior
Holiday vs non-holiday effects

Interpretation of Coefficients

Since the dependent variable is log-transformed, coefficients can be interpreted as approximate percentage changes in sales.

Key findings include:
- Temperature has a positive effect on sales, consistent with higher demand in warmer conditions
- Equipment availability shows the strongest and most stable positive impact
- Visits are significant in extended models, but their interpretation is affected by potential endogeneity

Visits Effect Interpretation

The effect of visits is sensitive to model specification.

While insignificant in baseline models, it becomes more pronounced in extended specifications. However, potential endogeneity implies that visits may be correlated with unobserved store performance.

# 10. CONCLUSION

This study identifies key drivers of FMCG beverage sales using statistical and econometric methods.

The results suggest that:
- Store execution quality is the most important driver of sales
- Weather conditions significantly affect demand patterns
- Store visits show mixed effects depending on model specification

From a business perspective, the findings highlight the importance of combining operational execution with environmental awareness in demand planning.

Overall, the analysis demonstrates how combining statistical inference with econometric modeling provides deeper insights into sales dynamics.